# Clean up the web-search blocklist lab

Remove the dedicated lab API and, if this lab created it, the backend. Keep the shared APIM service, resource group, reused backend, Foundry account, and model deployments. APIM identities and Foundry role assignments are also retained because other APIs may share them.

Use the same environment file and API name as the main notebook. This notebook reads the deployment record to identify whether a backend was created, so changing `BACKEND_ID` after deployment does not change the cleanup target.

The regex `/api/redact` function shares the proxy Function App and is removed with it; there is no additional redaction plan or storage account.

In [ ]:
from src.lab import load_config, service_parts, az_json

# Use the same selection as in the main notebook.
env_file = None
config = load_config(env_file)
service_id, subscription, resource_group, service_name = service_parts(config)
deployment_name = config.get("APIM_API_NAME", "web-search-blocklist")
deployment = az_json(
    "deployment", "group", "show", "--subscription", subscription,
    "--resource-group", resource_group, "--name", deployment_name,
)
parameters = deployment["properties"]["parameters"]
api_name = parameters["apiName"]["value"]
if parameters["apimServiceName"]["value"] != service_name:
    raise ValueError("The selected deployment belongs to a different APIM service.")
backend_created = not parameters["backendId"]["value"]
print("API to delete:", api_name)
print("Lab backend to delete:", f"{api_name}-foundry" if backend_created else "None; existing backend was reused")
proxy_backend_created = bool(parameters.get("streamingProxyUrl", {}).get("value"))
proxy_resources = {}
proxy_app_ids = set()
# Keep legacy resources discoverable after an upgrade to the storage-free app.
for suffix in ("-proxy", "-proxy-app", "-proxy-function"):
    try:
        proxy_deployment = az_json(
            "deployment", "group", "show", "--subscription", subscription,
            "--resource-group", resource_group, "--name", api_name + suffix,
        )
        outputs = {key: value["value"] for key, value in proxy_deployment["properties"].get("outputs", {}).items()}
        proxy_app_ids.update(outputs[key] for key in ("proxyAppId", "functionAppId") if key in outputs)
        proxy_resources.update(outputs)
    except RuntimeError as error:
        if "DeploymentNotFound" not in str(error):
            raise
print("Streaming proxy backend to delete:", f"{api_name}-streaming-proxy" if proxy_backend_created else "None")
for resource_id in sorted(proxy_app_ids):
    print("Proxy app to delete:", resource_id)
for key in ("planId", "storageId", "reporterSubscriptionId", "usageApiId"):
    if key in proxy_resources:
        print("Proxy resource to delete:", proxy_resources[key])
for resource in proxy_resources.get("networkResources", []):
    print("Proxy network resource to delete:", resource["id"])


## Delete the listed lab resources

Run the next cell after checking the API and backend names above. It does not delete the resource group or shared service. The deployment record is retained for auditing.

This also deletes the listed streaming Function/web apps, B1 plan, reporting subscription, and usage API. If the previous Function version is still present, it removes that Function, storage account, and dedicated network resources too; allow any old queued reports to drain first. Deployment records and shared Azure Monitor settings are retained.


In [ ]:
management_url = f"https://management.azure.com{service_id}"
az_json(
    "rest", "--method", "delete", "--url",
    f"{management_url}/apis/{api_name}?api-version=2024-05-01&deleteRevisions=true",
    "--headers", "If-Match=*",
)
if backend_created:
    az_json(
        "rest", "--method", "delete", "--url",
        f"{management_url}/backends/{api_name}-foundry?api-version=2024-05-01",
        "--headers", "If-Match=*",
    )
print("Deleted the lab API and any lab-created backend. Shared resources retained.")
if proxy_backend_created:
    az_json("rest", "--method", "delete", "--url",
            f"{management_url}/backends/{api_name}-streaming-proxy?api-version=2024-05-01",
            "--headers", "If-Match=*")
# Already-retired legacy resources can remain in the old deployment record.
def delete_if_present(*args):
    try:
        az_json(*args)
    except RuntimeError as error:
        if "ResourceNotFound" in str(error) or "NotFound" in str(error):
            return
        # CLI can report a polling error after Azure has completed deletion.
        if args[:2] == ("resource", "delete"):
            resource_id = args[args.index("--ids") + 1]
            version = args[args.index("--api-version") + 1]
            try:
                az_json("resource", "show", "--ids", resource_id, "--api-version", version)
            except RuntimeError as check:
                if "ResourceNotFound" in str(check) or "NotFound" in str(check):
                    return
                raise
        raise

# Delete every recorded app before the shared plan and any legacy network.
for resource_id in sorted(proxy_app_ids):
    delete_if_present("resource", "delete", "--ids", resource_id, "--api-version", "2024-04-01")
if "planId" in proxy_resources:
    delete_if_present("resource", "delete", "--ids", proxy_resources["planId"], "--api-version", "2024-04-01")
for resource in proxy_resources.get("networkResources", []):
    delete_if_present("resource", "delete", "--ids", resource["id"], "--api-version", resource["apiVersion"])
for key, version in (
    ("storageId", "2023-05-01"), ("reporterSubscriptionId", "2024-05-01"),
    ("usageApiId", "2024-05-01"),
):
    if key in proxy_resources:
        suffix = "&deleteRevisions=true" if key == "usageApiId" else ""
        delete_if_present("rest", "--method", "delete", "--url",
                f"https://management.azure.com{proxy_resources[key]}?api-version={version}{suffix}",
                "--headers", "If-Match=*")
print("Deleted the proxy app, plan, reporting API/subscription, and any remaining legacy Function/storage/network resources.")
